In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

C:\Users\gandh\AppData\Local\Temp\ipykernel_11428\1873884979.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\gandh\AppData\Local\Temp\ipykernel_11428\1873884979.py:6: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

In [4]:
loader = PyPDFLoader("data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [5]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [12]:
chunks = chunks[:3]

In [7]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"]
)

In [8]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [13]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

3 graph documents extracted
Nodes: ['Elon Musk', 'Pretoria', 'South Africa']
Rels:  [('Elon Musk', 'BORN_IN', 'Pretoria'), ('Elon Musk', 'BORN_IN', 'South Africa'), ('Elon Musk', 'HAS_NATIONALITY', 'American')]


In [14]:
print("Nodes:", [n.id for n in graph_docs[1].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[1].relationships])

Nodes: ['Elon Musk', 'Errol Musk', 'Maye Musk', 'Kimbal Musk', 'Tosca Musk', 'South Africa', 'Canada', 'Los Angeles', 'United States']
Rels:  [('Elon Musk', 'FATHER', 'Errol Musk'), ('Errol Musk', 'NATIONALITY', 'South Africa'), ('Elon Musk', 'MOTHER', 'Maye Musk'), ('Maye Musk', 'NATIONALITY', 'Canada'), ('Maye Musk', 'NATIONALITY', 'South Africa'), ('Elon Musk', 'BROTHER', 'Kimbal Musk'), ('Elon Musk', 'SISTER', 'Tosca Musk'), ('Tosca Musk', 'BASED_IN', 'Los Angeles'), ('Los Angeles', 'LOCATED_IN', 'United States')]


In [15]:
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)

print("Graph stored in Neo4J")

[#E928]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv6Address(('64:ff9b::23c8:9e8d', 7687, 0, 0)) (ResolvedIPv6Address(('64:ff9b::23c8:9e8d', 7687, 0, 0))): OSError('No data')
[#E929]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-f3cf1c0d-50a6-0001.production-orch-0894.neo4j.io', 7687)) (ResolvedIPv6Address(('64:ff9b::23c8:9e8d', 7687, 0, 0))): OSError('No data')
Transaction failed and will be retried in 1.1928624481981718s (Failed to read from defunct connection IPv4Address(('p-f3cf1c0d-50a6-0001.production-orch-0894.neo4j.io', 7687)) (ResolvedIPv6Address(('64:ff9b::23c8:9e8d', 7687, 0, 0))))


Graph stored in Neo4J


In [16]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [17]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 13}
  {'label': ['Document'], 'count': 3}
  {'label': ['__Entity__', 'Country'], 'count': 3}
  {'label': ['__Entity__', 'City'], 'count': 2}
  {'label': ['__Entity__'], 'count': 1}
Relationships:
  {'type': 'MENTIONS', 'count': 21}
  {'type': 'FATHER_OF', 'count': 6}
  {'type': 'MOTHER_OF', 'count': 6}
  {'type': 'NATIONALITY', 'count': 3}
  {'type': 'BORN_IN', 'count': 2}
  {'type': 'HAS_NATIONALITY', 'count': 1}
  {'type': 'FATHER', 'count': 1}
  {'type': 'MOTHER', 'count': 1}
  {'type': 'BROTHER', 'count': 1}
  {'type': 'SISTER', 'count': 1}
  {'type': 'MARRIED_TO', 'count': 1}
  {'type': 'BASED_IN', 'count': 1}
  {'type': 'LOCATED_IN', 'count': 1}
  {'type': 'FORMER_NAME_OF', 'count': 1}
